# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates the exploration and processing of the FAIR² "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and can be loaded via its URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset's Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review the available RecordSets, their `@id`s, and examine their fields and columns.

Entities should always be referenced by their `@id`.

In [ ]:
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}, dataType={field.data_type})")
    print(f"  Columns:")
    for col in rs.columns:
        print(f"    - {col.name} (@id={col.id}) (field: {col.field.id if col.field else None})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the RecordSet and field `@id`s determined above.

In [ ]:
# Select the main RecordSet for analysis -- in this dataset, it is likely the one containing protein measurements.
# You may list them for review if unsure.
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Available RecordSets @id:")
for rs in metadata.record_sets:
    print(f"  - {rs.name}: {rs.id}")

# We'll choose the first (or main) RecordSet for demonstration. Please adjust as needed for your tasks.
target_record_set_id = record_set_ids[0]

# Extract data from all record sets for generality
dataframes = {}
for rs_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df

print("\nColumns for RecordSet @id:", target_record_set_id)
print(dataframes[target_record_set_id].columns.tolist())
dataframes[target_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Select and process numeric fields for analysis, filter outliers, normalize values, and group data as appropriate using the field `@id`s.

> **Tip:** Replace placeholder variable values for `numeric_field_id` and `group_field_id` below based on the overviews above if you want to analyze other fields.

In [ ]:
# Choose a numeric field to analyze. Let's use the first numeric field available.
df = dataframes[target_record_set_id]

# Identify numeric field IDs by checking data types in the RecordSet.
target_rs = None
for rs in metadata.record_sets:
    if rs.id == target_record_set_id:
        target_rs = rs
        break

numeric_fields = [f for f in target_rs.fields if f.data_type in ('Float', 'Number', 'Integer')]
if numeric_fields:
    numeric_field = numeric_fields[0]
    numeric_field_id = numeric_field.id
    print(f"Numeric field selected: {numeric_field.name} (@id={numeric_field_id})")
else:
    print("No numeric fields found in this record set.")

# Pick a grouping field - as an example, we use the first non-numeric field that is not unique.
group_fields = [f for f in target_rs.fields if f.data_type == 'Text']
if group_fields:
    group_field = group_fields[0]
    group_field_id = group_field.id
    print(f"Grouping field selected: {group_field.name} (@id={group_field_id})")
else:
    group_field = None
    group_field_id = None

if numeric_fields:
    # Prepare field name for DataFrame (the '@id')
    threshold = df[numeric_field_id].mean() if numeric_field_id in df and pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    # Only keep records above the mean (or above an arbitrary value if NaN)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (z-score):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the selected field, if it's available and makes sense
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships in the data using the selected fields.
You may choose different fields for more advanced plots if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field.name} (@id={numeric_field_id})")
    plt.xlabel(numeric_field.name)
    plt.show()

    # If grouping field available, plot group means
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(12,5))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Group mean of {numeric_field.name} by {group_field.name}")
        plt.xlabel(group_field.name)
        plt.ylabel(f"Mean {numeric_field.name}")
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion

Using the FAIR² dataset, we loaded protein abundance and related fields for extracellular vesicles from human mast cells, examined the fields and their types, filtered and normalized on a numerical field, grouped by a categorical field, and visualized the distributions.

The approach shown here can be adapted for in-depth proteomics or bioinformatics analysis by choosing relevant fields and record sets using their unique `@id`s.